# Asistente RAG sobre Hades (2018)

Asistente documental que responde preguntas en inglés sobre el videojuego Hades (Supergiant Games) usando solo una base de conocimiento propia, con memoria de conversación.

Flujo: documentos → embeddings → Qdrant → recuperación (denso + BM25) → contexto → LLM → respuesta con fuentes.

- Base documental: hades1.json (wiki de Hades en Fandom, CC BY-SA, limpiada con el scraper) y builds.json (builds escritas por el grupo).
- Embeddings: BAAI/bge-base-en-v1.5.
- Base vectorial: Qdrant en modo local.
- Modelo generativo: Qwen/Qwen2.5-3B-Instruct (Hugging Face).
- Memoria: historial de turnos y reescritura de las preguntas de seguimiento.

Se ejecuta en Google Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → T4).

## 1. Preparar el entorno

Se fija la misma versión de Transformers que en el notebook de la práctica. Si es la primera instalación, reinicia la sesión (Entorno de ejecución → Reiniciar sesión) y continúa desde la celda de configuración.

In [ ]:
%pip install -q "transformers==4.57.6" sentence-transformers qdrant-client rank-bm25 accelerate pandas

In [ ]:
import os, re, json
os.environ["USE_TF"] = "0"

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from qdrant_client import QdrantClient, models
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 200)

CONFIG = {
    "ruta_hades": "hades1.json",
    "ruta_builds": "builds.json",            # opcional: builds escritas por el grupo
    "modelo_embeddings": "BAAI/bge-base-en-v1.5",
    "prefijo_consulta": "Represent this sentence for searching relevant passages: ",
    "modelo_llm": "Qwen/Qwen2.5-3B-Instruct",
    "coleccion": "hades1",
    "ruta_qdrant": "qdrant_hades",
    "k": 5,
    "turnos_memoria": 4,
}
NO_INFO = "I can't find that information in the available documents."

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU disponible: True
Tesla T4


## 2. Subir los archivos

Sube hades1.json (y builds.json si ya lo tenéis) al entorno de Colab.

In [ ]:
if not os.path.exists(CONFIG["ruta_hades"]):
    from google.colab import files
    print("Sube hades1.json (y, si existe, builds.json)")
    files.upload()

print("hades1.json:", os.path.exists(CONFIG["ruta_hades"]))
print("builds.json:", os.path.exists(CONFIG["ruta_builds"]))

hades1.json: True
builds.json: True


## 3. Base documental

Cada fragmento tiene entidad, sección, categoría, tipo (juego, historia o trivia), fuente, url y texto. El texto empieza por el nombre de la entidad para que se entienda por sí solo cuando se recupera aislado.

Al cargar, se eliminan las frases que mencionan Hades II o Melinoë (la wiki mezcla los dos juegos), los fragmentos demasiado cortos y los duplicados. Las builds del grupo se añaden desde builds.json; las entradas de la plantilla sin rellenar (texto que empieza por corchete) se ignoran.

In [ ]:
PATRON_HADES2 = re.compile(r"Melino[eë]|Hades II", re.IGNORECASE)

def quitar_hades2(doc):
    original = doc["texto"].strip()
    lineas = []
    for linea in original.split("\n"):
        frases = re.split(r"(?<=[.!?])\s+", linea)
        frases = [f for f in frases if not PATRON_HADES2.search(f)]
        if frases:
            lineas.append(" ".join(frases))
    limpio = "\n".join(lineas).strip()
    # si se perdió la cabecera "Entidad - Sección:", se restituye
    if limpio and limpio != original and not limpio.startswith(doc["entidad"]):
        limpio = f"{doc['entidad']} - {doc.get('seccion', '')}: {limpio}"
    return limpio

def cargar_documentos():
    with open(CONFIG["ruta_hades"], encoding="utf-8") as f:
        crudos = json.load(f)

    if os.path.exists(CONFIG["ruta_builds"]):
        with open(CONFIG["ruta_builds"], encoding="utf-8") as f:
            for b in json.load(f):
                if str(b.get("texto", "")).strip().startswith("["):
                    continue
                b.setdefault("entidad", "Build")
                b.setdefault("categoria", "build")
                b.setdefault("tipo", "juego")
                b.setdefault("seccion", "Build")
                b.setdefault("fuente", "grupo (propio)")
                b.setdefault("url", "")
                crudos.append(b)

    docs, vistos, descartados = [], set(), 0
    for d in crudos:
        texto = quitar_hades2(d)
        if len(texto.split()) < 12 or texto in vistos:
            descartados += 1
            continue
        vistos.add(texto)
        docs.append({
            "id": len(docs),
            "entidad": d["entidad"],
            "seccion": d.get("seccion", ""),
            "categoria": d.get("categoria", "general"),
            "tipo": d.get("tipo", "juego"),
            "fuente": d.get("fuente", ""),
            "url": d.get("url", ""),
            "texto": texto,
        })
    print(f"Fragmentos cargados: {len(docs)} (descartados: {descartados})")
    return docs

DOCS = cargar_documentos()

Fragmentos cargados: 1889 (descartados: 7)


In [ ]:
df_docs = pd.DataFrame(DOCS)
palabras = df_docs["texto"].str.split().str.len()

display(df_docs.groupby("categoria")["tipo"].value_counts().unstack(fill_value=0))
print("Entidades distintas:", df_docs["entidad"].nunique())
print("Palabras por fragmento (mediana / máximo):", palabras.median(), "/", palabras.max())
print("Fragmentos que aún mencionan Hades II o Melinoë:", int(df_docs["texto"].str.contains(PATRON_HADES2).sum()))

tipo,historia,juego,trivia
categoria,,,
arma,0,118,8
aspecto,162,117,62
bendicion,0,79,0
build,0,32,0
enemigo,2,69,3
general,0,99,0
jefe,34,117,20
martillo,0,3,0
mecanica,0,108,0


Entidades distintas: 163
Palabras por fragmento (mediana / máximo): 76.0 / 210
Fragmentos que aún mencionan Hades II o Melinoë: 0


## 4. Embeddings

Cada fragmento se convierte en un vector con bge-base-en-v1.5 (768 dimensiones). Los vectores se normalizan, de modo que el producto escalar equivale a la similitud coseno. A las preguntas se les antepone el prefijo de instrucción que recomienda el modelo; a los fragmentos no.

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_embeddings = SentenceTransformer(CONFIG["modelo_embeddings"], device=dispositivo)

EMB = modelo_embeddings.encode(
    [d["texto"] for d in DOCS],
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Matriz de embeddings:", EMB.shape)

def embedding_pregunta(pregunta):
    return modelo_embeddings.encode(
        CONFIG["prefijo_consulta"] + pregunta,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Matriz de embeddings: (1889, 768)


## 5. Qdrant

Los vectores se guardan en una colección de Qdrant en modo local (una carpeta, sin servidor). El payload de cada punto contiene todos los campos del fragmento, así que se puede filtrar por tipo o categoría.

Si se reejecuta la celda, se cierra el cliente anterior para evitar el bloqueo del modo local y se recrea la colección. En Colab el disco se borra al terminar la sesión; el índice se reconstruye en segundos a partir de los JSON.

In [ ]:
try:
    cliente_qdrant.close()
except NameError:
    pass

cliente_qdrant = QdrantClient(path=CONFIG["ruta_qdrant"])

if cliente_qdrant.collection_exists(CONFIG["coleccion"]):
    cliente_qdrant.delete_collection(CONFIG["coleccion"])

cliente_qdrant.create_collection(
    collection_name=CONFIG["coleccion"],
    vectors_config=models.VectorParams(size=EMB.shape[1], distance=models.Distance.COSINE),
)

for inicio in range(0, len(DOCS), 256):
    lote = range(inicio, min(inicio + 256, len(DOCS)))
    cliente_qdrant.upsert(
        collection_name=CONFIG["coleccion"],
        points=[models.PointStruct(id=i, vector=EMB[i].tolist(), payload=DOCS[i]) for i in lote],
    )

print("Puntos en Qdrant:", cliente_qdrant.count(collection_name=CONFIG["coleccion"]).count)

Puntos en Qdrant: 1889


## 6. Recuperación

Hay tres piezas:

- Búsqueda densa en Qdrant: encuentra fragmentos con significado parecido a la pregunta.
- Búsqueda BM25: encuentra coincidencias exactas de palabras, útil con nombres propios del juego como Heart-Seeking Bow o Titan Blood.
- Fusión RRF: combina los dos rankings por posición, sin tener que calibrar sus puntuaciones.

Todas las búsquedas filtran por tipo. Por defecto solo se usan fragmentos de tipo juego, para que la historia y la trivia no compitan con los datos de juego.

In [ ]:
TODOS_LOS_TIPOS = ("juego", "historia", "trivia")

def tokenizar(texto):
    return re.findall(r"[a-z0-9']+", texto.lower())

BM25 = BM25Okapi([tokenizar(d["texto"]) for d in DOCS])
TIPOS_DOC = np.array([d["tipo"] for d in DOCS])

def buscar_denso(vector, n, tipos):
    filtro = models.Filter(must=[
        models.FieldCondition(key="tipo", match=models.MatchAny(any=list(tipos)))
    ])
    respuesta = cliente_qdrant.query_points(
        collection_name=CONFIG["coleccion"],
        query=vector.tolist(),
        limit=n,
        query_filter=filtro,
        with_payload=False,
    )
    return [p.id for p in respuesta.points]

def buscar_bm25(pregunta, n, tipos):
    puntos = BM25.get_scores(tokenizar(pregunta))
    puntos = np.where(np.isin(TIPOS_DOC, list(tipos)), puntos, -1.0)
    orden = np.argsort(puntos)[::-1][:n]
    return [int(i) for i in orden if puntos[i] > 0]

def fusion_rrf(listas, c=60):
    acumulado = {}
    for lista in listas:
        for posicion, idx in enumerate(lista):
            acumulado[idx] = acumulado.get(idx, 0.0) + 1.0 / (c + posicion + 1)
    return sorted(acumulado, key=acumulado.get, reverse=True)

def recuperar(pregunta, k=None, modo="hibrido", tipos=("juego",)):
    k = k or CONFIG["k"]
    vector = embedding_pregunta(pregunta)
    if modo == "denso":
        ids = buscar_denso(vector, k, tipos)
    else:
        candidatos = max(30, 4 * k)
        ids = fusion_rrf([
            buscar_denso(vector, candidatos, tipos),
            buscar_bm25(pregunta, candidatos, tipos),
        ])[:k]
    return [{"documento": DOCS[i], "puntuacion": float(EMB[i] @ vector)} for i in ids]

def mostrar_recuperacion(pregunta, **kwargs):
    for pos, r in enumerate(recuperar(pregunta, **kwargs), start=1):
        d = r["documento"]
        print(f"{pos}. [{d['id']}] {d['entidad']} - {d['seccion']} | similitud {r['puntuacion']:.3f}")
        print("   ", d["texto"][:170].replace("\n", " "))

mostrar_recuperacion("Which aspects does the Heart-Seeking Bow have?")

1. [58] Heart-Seeking Bow - Aspects | similitud 0.776
    Heart-Seeking Bow - Aspects: The Heart-Seeking Bow has 4 Aspects: Zagreus, Chiron, Hera, and Rama. Like all other weapons, each Aspect has 5 levels that can be unlocked b
2. [59] Heart-Seeking Bow - Aspects | similitud 0.764
    Heart-Seeking Bow - Aspects: Aspect | Description | Upgrade | Cost | Appearance Aspect of Zagreus | The form in which the heart-seeking bow first revealed itself. It sens
3. [55] Heart-Seeking Bow - Resumen | similitud 0.671
    Heart-Seeking Bow: Coronacht (also known as Coronacht, the Heart-Seeking Bow) is one of the Infernal Arms in Hades and can be unlocked for 1 Chthonic Key.
4. [60] Heart-Seeking Bow - Aspects | similitud 0.693
    Heart-Seeking Bow - Aspects: Aspect | Description | Upgrade | Cost | Appearance Aspect of Chiron | Your Special automatically seeks the foe last struck by your Attack. Wh
5. [63] Heart-Seeking Bow - Revealing the Aspect of Rama | similitud 0.653
    Heart-Seeking Bow - R

## 7. Modelo generativo y prompt

Qwen2.5-3B-Instruct es un modelo de chat, así que se usa su plantilla de mensajes (sistema, usuario, asistente). El prompt de sistema le obliga a responder solo con los fragmentos numerados, a citar sus ids y a usar una frase fija cuando no hay información, lo que permite detectar la abstención automáticamente.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["modelo_llm"])
llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["modelo_llm"], torch_dtype=torch.float16, device_map="auto"
)
llm.eval()

def generar(mensajes, max_new_tokens=300):
    texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(texto, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        salida = llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
    nuevos = salida[0][entradas["input_ids"].shape[1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()

SYSTEM_RAG = (
    "You are an expert assistant for the video game Hades (2018) by Supergiant Games. "
    "Answer the question using ONLY the numbered CONTEXT passages. "
    f"If the context does not contain the answer, reply exactly: {NO_INFO} "
    "Do not use outside knowledge and do not talk about Hades II. "
    "Be concise (at most 5 sentences) and cite the ids of the passages you used, like [12] [40]."
)

def construir_contexto(resultados):
    return "\n\n".join(f"[{r['documento']['id']}] {r['documento']['texto']}" for r in resultados)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 8. Asistente con memoria

La clase AsistenteHades une recuperación y generación y mantiene el historial de la conversación.

La memoria funciona en dos niveles:

- Reescritura de la pregunta: una pregunta como "And what does the one named after Hera do?" no sirve para buscar. Antes de recuperar, el modelo la convierte en una pregunta autónoma usando el historial, y la búsqueda se hace con esa.
- Historial en el prompt: los últimos turnos (pregunta y respuesta) se pasan como mensajes del chat para que la respuesta mantenga el hilo.

Parámetros para comparar variantes: modo (denso o híbrido), tipos, k, umbral de similitud mínima y memoria. Si la mejor similitud no llega al umbral, el asistente se abstiene sin llamar al modelo.

In [ ]:
SYSTEM_CONDENSAR = (
    "Rewrite the user's last question as a standalone question, using the conversation to resolve "
    "pronouns and references such as 'it', 'that one' or 'its'. Keep game names exactly as written. "
    "Output only the rewritten question and nothing else."
)

class AsistenteHades:
    def __init__(self, modo="hibrido", tipos=("juego",), k=None, umbral=None, memoria=True, turnos=None):
        self.modo = modo
        self.tipos = tipos
        self.k = k or CONFIG["k"]
        self.umbral = umbral
        self.memoria = memoria
        self.turnos = turnos or CONFIG["turnos_memoria"]
        self.historial = []

    def reiniciar(self):
        self.historial = []

    def _autonoma(self, pregunta):
        if not (self.memoria and self.historial):
            return pregunta
        conversacion = "\n".join(f"User: {p}\nAssistant: {r}" for p, r in self.historial[-self.turnos:])
        mensajes = [
            {"role": "system", "content": SYSTEM_CONDENSAR},
            {"role": "user", "content": f"Conversation:\n{conversacion}\n\nLast question: {pregunta}"},
        ]
        nueva = generar(mensajes, max_new_tokens=60).split("\n")[0].strip()
        return nueva if 5 <= len(nueva) <= 300 else pregunta

    def preguntar(self, pregunta):
        autonoma = self._autonoma(pregunta)
        resultados = recuperar(autonoma, k=self.k, modo=self.modo, tipos=self.tipos)
        mejor = max((r["puntuacion"] for r in resultados), default=0.0)

        if not resultados or (self.umbral is not None and mejor < self.umbral):
            respuesta = NO_INFO
        else:
            mensajes = [{"role": "system", "content": SYSTEM_RAG}]
            if self.memoria:
                for p, r in self.historial[-self.turnos:]:
                    mensajes += [{"role": "user", "content": p}, {"role": "assistant", "content": r}]
            mensajes.append({
                "role": "user",
                "content": f"CONTEXT:\n{construir_contexto(resultados)}\n\nQUESTION: {autonoma}",
            })
            respuesta = generar(mensajes)

        if self.memoria:
            self.historial.append((pregunta, respuesta))

        return {
            "pregunta": pregunta,
            "pregunta_autonoma": autonoma,
            "respuesta": respuesta,
            "recuperados": resultados,
            "mejor_similitud": mejor,
        }

def mostrar(r):
    print("QUESTION:", r["pregunta"])
    if r["pregunta_autonoma"] != r["pregunta"]:
        print("STANDALONE QUESTION:", r["pregunta_autonoma"])
    print("ANSWER:", r["respuesta"])
    print("SOURCES:")
    for x in r["recuperados"]:
        d = x["documento"]
        print(f"  [{d['id']}] {d['entidad']} - {d['seccion']} ({x['puntuacion']:.3f}) {d['url']}")

## 9. Pruebas

Se definen 12 preguntas con respuesta en la colección y 3 sin respuesta. Cada pregunta con respuesta lleva palabras clave que deben aparecer en el fragmento que la contiene; sirven para medir la recuperación de forma automática. Las preguntas sin respuesta incluyen una sobre Hades II, otra sobre ventas y otra sobre requisitos técnicos.

In [ ]:
PRUEBAS_DENTRO = [
    ("Which aspects does the Stygian Blade have?", ["Nemesis", "Poseidon", "Arthur"]),
    ("What does the Aspect of Arthur give you?", ["Excalibur", "+50"]),
    ("What does the Aspect of Nemesis do for the Stygian Blade?", ["Critical", "Special"]),
    ("Which aspects does the Heart-Seeking Bow have?", ["Chiron", "Hera", "Rama"]),
    ("What does the Aspect of Hera do for the Heart-Seeking Bow?", ["Cast loads into your next Attack"]),
    ("What does the Hard Labor condition do in the Pact of Punishment?", ["Hard Labor", "+20%"]),
    ("How many encounters does a keepsake need to reach rank 2?", ["25 encounters"]),
    ("How many tasks are there in the Fated List of Minor Prophecies?", ["55 tasks"]),
    ("What are Duo Boons?", ["combine the powers of two different gods"]),
    ("How much damage does the Bone Hydra's Lunge deal below 66% health?", ["Lunge", "increases to 16"]),
    ("How can I get Darkness?", ["Infernal Troves"]),
    ("What happens when you pick one god's boon in a Trial of the Gods?", ["spurned"]),
]

PRUEBAS_FUERA = [
    "What is Melinoë's main weapon in Hades II?",
    "How many copies of Hades has Supergiant Games sold worldwide?",
    "Which graphics card do I need to play Hades at 4K?",
]

def contiene_claves(doc, claves):
    return all(c.lower() in doc["texto"].lower() for c in claves)

# Comprobación: cada pregunta con respuesta debe tener al menos un fragmento que contenga sus claves
for pregunta, claves in PRUEBAS_DENTRO:
    n = sum(contiene_claves(d, claves) for d in DOCS)
    print(f"{n:3d} fragmentos con las claves | {pregunta}")

  1 fragmentos con las claves | Which aspects does the Stygian Blade have?
  2 fragmentos con las claves | What does the Aspect of Arthur give you?
  9 fragmentos con las claves | What does the Aspect of Nemesis do for the Stygian Blade?
  1 fragmentos con las claves | Which aspects does the Heart-Seeking Bow have?
  2 fragmentos con las claves | What does the Aspect of Hera do for the Heart-Seeking Bow?
  2 fragmentos con las claves | What does the Hard Labor condition do in the Pact of Punishment?
  1 fragmentos con las claves | How many encounters does a keepsake need to reach rank 2?
  1 fragmentos con las claves | How many tasks are there in the Fated List of Minor Prophecies?
  1 fragmentos con las claves | What are Duo Boons?
  1 fragmentos con las claves | How much damage does the Bone Hydra's Lunge deal below 66% health?
  9 fragmentos con las claves | How can I get Darkness?
  5 fragmentos con las claves | What happens when you pick one god's boon in a Trial of the Gods?


In [ ]:
asistente = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=None, memoria=True)

for pregunta, _ in PRUEBAS_DENTRO[:5]:
    asistente.reiniciar()
    print("=" * 90)
    mostrar(asistente.preguntar(pregunta))

QUESTION: Which aspects does the Stygian Blade have?
ANSWER: The Stygian Blade has four aspects: Zagreus, Nemesis, Poseidon, and Arthur. [3]
SOURCES:
  [3] Stygian Blade - Aspects (0.796) https://hades.fandom.com/wiki/Stygian_Blade
  [4] Stygian Blade - Aspects (0.797) https://hades.fandom.com/wiki/Stygian_Blade
  [6] Stygian Blade - Aspects (0.763) https://hades.fandom.com/wiki/Stygian_Blade
  [5] Stygian Blade - Aspects (0.742) https://hades.fandom.com/wiki/Stygian_Blade
  [0] Stygian Blade - Resumen (0.730) https://hades.fandom.com/wiki/Stygian_Blade
QUESTION: What does the Aspect of Arthur give you?
ANSWER: The Aspect of Arthur gives you Holy Excalibur and +50 to your Life Total, and it changes your move set: the Heavy Attack and the Special create an aura that slows foes, strongly slows enemy projectiles and disables floor and wall traps. The aura lasts 8 seconds. [3] [1860] [1862]
SOURCES:
  [8] Stygian Blade - Revealing the Aspect of Arthur (0.680) https://hades.fandom.com/wiki/

In [ ]:
for pregunta in PRUEBAS_FUERA:
    asistente.reiniciar()
    print("=" * 90)
    mostrar(asistente.preguntar(pregunta))

QUESTION: What is Melinoë's main weapon in Hades II?
ANSWER: I can't find that information in the available documents. [0] [1202]
SOURCES:
  [0] Stygian Blade - Resumen (0.605) https://hades.fandom.com/wiki/Stygian_Blade
  [605] Hades - Hades (0.607) https://hades.fandom.com/wiki/Hades
  [70] Twin Fists - Resumen (0.627) https://hades.fandom.com/wiki/Twin_Fists
  [1338] House Contractor - Resumen (0.457) https://hades.fandom.com/wiki/House_Contractor
  [1202] Hades (game) - Official description (0.609) https://hades.fandom.com/wiki/Hades_%28game%29
QUESTION: How many copies of Hades has Supergiant Games sold worldwide?
ANSWER: I can't find that information in the available documents. [1201]
SOURCES:
  [1201] Hades (game) - Datos generales (0.689) https://hades.fandom.com/wiki/Hades_%28game%29
  [1202] Hades (game) - Official description (0.563) https://hades.fandom.com/wiki/Hades_%28game%29
  [1029] House of Hades - Administrative Chamber (0.522) https://hades.fandom.com/wiki/House_of_

## 10. Evaluación automática de la recuperación

Para cada pregunta con respuesta se comprueba si alguno de los K fragmentos recuperados contiene las palabras clave (hit@k) y en qué posición aparece el primero (MRR). Se comparan tres variantes de recuperación y varios valores de K.

In [ ]:

# Fragmentos relevantes revisados manualmente.
# Cada conjunto contiene los IDs que aportan información
# útil para responder a su pregunta correspondiente.

RELEVANTES_IDS = {
    PRUEBAS_DENTRO[0][0]: {3},
    PRUEBAS_DENTRO[1][0]: {1860},
    PRUEBAS_DENTRO[2][0]: {5, 1857, 1859},
    PRUEBAS_DENTRO[3][0]: {58},
    PRUEBAS_DENTRO[4][0]: {61, 1245},
    PRUEBAS_DENTRO[5][0]: {1639},
    PRUEBAS_DENTRO[6][0]: {871},
    PRUEBAS_DENTRO[7][0]: {337},
    PRUEBAS_DENTRO[8][0]: {1121, 1122},
    PRUEBAS_DENTRO[9][0]: {1286},
    PRUEBAS_DENTRO[10][0]: {398},
    PRUEBAS_DENTRO[11][0]: {1155},
}

# Comprobar que todas las preguntas tienen documentos de referencia
assert len(RELEVANTES_IDS) == len(PRUEBAS_DENTRO)

def evaluar_recuperacion(k=5, modo="hibrido", tipos=("juego",)):
    aciertos = 0
    rr = []

    for pregunta, claves in PRUEBAS_DENTRO:

        relevantes = RELEVANTES_IDS[pregunta]

        resultados = recuperar(
            pregunta,
            k=k,
            modo=modo,
            tipos=tipos
        )

        ids_recuperados = [
            r["documento"]["id"] for r in resultados
        ]

        posiciones = [
            i + 1
            for i, id_doc in enumerate(ids_recuperados)
            if id_doc in relevantes
        ]

        posicion = min(posiciones) if posiciones else None

        aciertos += posicion is not None
        rr.append(1 / posicion if posicion else 0.0)

    return {
        "hit@k": round(aciertos / len(PRUEBAS_DENTRO), 2),
        "MRR": round(float(np.mean(rr)), 2)
    }


# Comparar las tres configuraciones
variantes = [
    ("denso, todos los tipos", "denso", TODOS_LOS_TIPOS),
    ("denso, solo juego", "denso", ("juego",)),
    ("híbrido, solo juego", "hibrido", ("juego",)),
]

tabla_recuperacion = pd.DataFrame(
    [
        {
            "variante (k=5)": nombre,
            **evaluar_recuperacion(5, modo, tipos)
        }
        for nombre, modo, tipos in variantes
    ]
)

display(tabla_recuperacion)


# Evaluar diferentes cantidades de fragmentos recuperados
tabla_k = pd.DataFrame(
    [
        {
            "k": k,
            **evaluar_recuperacion(k, "hibrido", ("juego",))
        }
        for k in [1, 3, 5, 8]
    ]
)

display(tabla_k)


,variante (k=5),hit@k,MRR
0,"denso, todos los tipos",1.0,0.85
1,"denso, solo juego",1.0,0.85
2,"híbrido, solo juego",1.0,0.75


,k,hit@k,MRR
0,1,0.58,0.58
1,3,0.83,0.71
2,5,1.00,0.75
3,8,1.00,0.75


## 11. Umbral de similitud

Recuperar siempre devuelve los fragmentos más parecidos, aunque la respuesta no esté. Se compara la mejor similitud de las preguntas con respuesta con la de las preguntas sin respuesta. Solo se fija un umbral si los dos grupos se separan.

Limitación: las mismas 15 preguntas sirven para calibrar el umbral y para evaluar el sistema, y el test no debería usarse para ajustar. Por eso el umbral y las métricas de abstención son orientativos y se indica en las limitaciones.

In [ ]:
dentro = [recuperar(p, k=1, modo="denso")[0]["puntuacion"] for p, _ in PRUEBAS_DENTRO]
fuera = [recuperar(p, k=1, modo="denso")[0]["puntuacion"] for p in PRUEBAS_FUERA]

print(f"Con respuesta: mínimo {min(dentro):.3f}, media {np.mean(dentro):.3f}")
print(f"Sin respuesta: máximo {max(fuera):.3f}, media {np.mean(fuera):.3f}")

if max(fuera) < min(dentro):
    UMBRAL = round((max(fuera) + min(dentro)) / 2, 3)
    print("Los grupos se separan. Umbral propuesto:", UMBRAL)
else:
    UMBRAL = None
    print("Los grupos se solapan: la similitud sola no permite abstenerse de forma fiable.")
    print("Se confía en la instrucción de abstención del prompt.")

Con respuesta: mínimo 0.680, media 0.778
Sin respuesta: máximo 0.689, media 0.655
Los grupos se solapan: la similitud sola no permite abstenerse de forma fiable.
Se confía en la instrucción de abstención del prompt.


## 12. Evaluación manual

La evaluación sigue el esquema de evaluación de un RAG: se juzgan por separado la recuperación (¿llegan al contexto los fragmentos adecuados?) y la generación (¿la respuesta usa bien esos fragmentos?).

Cómo se hace:

1. Se lanzan las 15 preguntas (12 con respuesta y 3 sin respuesta) con el sistema final y se guarda la tabla evaluacion.
2. Se leen las fichas de revisión: pregunta, respuesta y fragmentos recuperados, con los citados marcados.
3. Se puntúa cada criterio de 1 a 5 y se escribe una observación en la celda de puntuaciones.
4. La celda de revisión comprueba que están las 15 preguntas, los seis criterios y las observaciones.

Rúbrica (5 = muy bien, 3 = aceptable, 1 = mal; NA cuando el criterio no se aplica):

| Criterio | Qué se juzga | 5 | 3 | 1 |
|---|---|---|---|---|
| recuperacion | Context relevance: el contexto es adecuado para la pregunta | El fragmento con la respuesta está entre los primeros y el resto es relevante | Está entre los K pero con mucho ruido | No está |
| fidelidad | Faithfulness: lo que se afirma está respaldado por el contexto | Todo está respaldado | Cambia o añade algún detalle | Inventa |
| correccion | La respuesta coincide con lo que dice la wiki | Exacta y completa | Parcial o con un error menor | Incorrecta |
| relevancia | Answer relevance: responde a lo que se pregunta | Responde justo a la pregunta | Responde en parte o con información de más | Habla de otra cosa |
| trazabilidad | Los ids citados respaldan lo afirmado | Todos los ids contienen lo que se afirma | Algún id no corresponde | Sin citas o ids incorrectos |
| abstencion | Alucinaciones (solo preguntas sin respuesta) | Dice la frase fija y nada más | Se abstiene pero añade información | Inventa una respuesta |

Criterios que no se aplican (NA): en las preguntas sin respuesta, recuperacion, correccion, relevancia y trazabilidad; en las preguntas con respuesta, abstencion. En las preguntas sin respuesta, fidelidad se puntúa según si el sistema evita inventar.

Las columnas ids citados, citas en el contexto, cita contiene las claves y claves en la respuesta son ayudas automáticas aproximadas; la puntuación la decide quien lee la respuesta y los fragmentos.

In [ ]:
def abstiene(respuesta):
    return "can't find" in respuesta.replace("’", "'").lower()

def ids_citados(respuesta):
    return [int(x) for x in re.findall(r"\[(\d+)\]", respuesta)]

asistente = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=UMBRAL, memoria=True)

casos = [(p, c, "con respuesta") for p, c in PRUEBAS_DENTRO] + [(p, None, "sin respuesta") for p in PRUEBAS_FUERA]
RESULTADOS, filas = [], []

for pregunta, claves, esperada in casos:
    asistente.reiniciar()
    r = asistente.preguntar(pregunta)
    RESULTADOS.append(r)
    recuperados = [x["documento"]["id"] for x in r["recuperados"]]
    citados = ids_citados(r["respuesta"])
    filas.append({
        "pregunta": pregunta,
        "esperada": esperada,
        "respuesta": r["respuesta"],
        "fuentes (ids)": recuperados,
        "ids citados": citados,
        "citas en el contexto": (bool(citados) and all(i in recuperados for i in citados)) if claves else "-",
        "cita contiene las claves": any(contiene_claves(DOCS[i], claves) for i in citados if i < len(DOCS)) if claves else "-",
        "claves en la respuesta": f"{sum(c.lower() in r['respuesta'].lower() for c in claves)}/{len(claves)}" if claves else "-",
        "se abstiene": abstiene(r["respuesta"]),
    })

evaluacion = pd.DataFrame(filas)
for col in ["recuperacion (1-5)", "fidelidad (1-5)", "correccion (1-5)", "relevancia (1-5)", "trazabilidad (1-5)", "abstencion (1-5)"]:
    evaluacion[col] = None
evaluacion["observaciones"] = ""

print("Preguntas evaluadas:", len(evaluacion))
display(evaluacion[["pregunta", "esperada", "ids citados", "citas en el contexto", "cita contiene las claves", "claves en la respuesta", "se abstiene"]])

Preguntas evaluadas: 15


,pregunta,esperada,ids citados,citas en el contexto,cita contiene las claves,claves en la respuesta,se abstiene
0,Which aspects does the Stygian Blade have?,con respuesta,[3],True,True,3/3,False
1,What does the Aspect of Arthur give you?,con respuesta,"[3, 1860, 1862]",True,True,2/2,False
2,What does the Aspect of Nemesis do for the Stygian Blade?,con respuesta,"[5, 1859]",True,True,2/2,False
3,Which aspects does the Heart-Seeking Bow have?,con respuesta,[58],True,True,3/3,False
4,What does the Aspect of Hera do for the Heart-Seeking Bow?,con respuesta,[1245],True,False,0/1,False
5,What does the Hard Labor condition do in the Pact of Punishment?,con respuesta,[1639],True,True,1/2,False
6,How many encounters does a keepsake need to reach rank 2?,con respuesta,[871],True,True,1/1,False
7,How many tasks are there in the Fated List of Minor Prophecies?,con respuesta,[337],True,True,1/1,False
8,What are Duo Boons?,con respuesta,"[1121, 1122]",True,True,1/1,False
9,How much damage does the Bone Hydra's Lunge deal below 66% health?,con respuesta,[1286],True,True,1/2,False


### Fichas de revisión

Cada ficha muestra la pregunta, la respuesta y los cinco fragmentos recuperados. Un asterisco marca los que la respuesta cita. Para volver a ver un caso suelto, usa ver_caso(i) con i de 0 a 14.

In [ ]:
def ver_caso(i, caracteres=260):
    r = RESULTADOS[i]
    fila = evaluacion.loc[i]
    print("=" * 100)
    print(f"PREGUNTA {i + 1} ({fila['esperada']}): {fila['pregunta']}")
    print("RESPUESTA:", fila["respuesta"])
    print(f"ids citados: {fila['ids citados']} | claves en la respuesta: {fila['claves en la respuesta']} | se abstiene: {fila['se abstiene']}")
    print("FRAGMENTOS RECUPERADOS (* = citado en la respuesta):")
    for pos, x in enumerate(r["recuperados"], start=1):
        d = x["documento"]
        marca = "*" if d["id"] in fila["ids citados"] else " "
        print(f" {marca}{pos}. [{d['id']}] {d['entidad']} - {d['seccion']} (similitud {x['puntuacion']:.3f})")
        print("      ", d["texto"][:caracteres].replace("\n", " "))

for i in range(len(evaluacion)):
    ver_caso(i)

PREGUNTA 1 (con respuesta): Which aspects does the Stygian Blade have?
RESPUESTA: The Stygian Blade has four aspects: Zagreus, Nemesis, Poseidon, and Arthur. [3]
ids citados: [3] | claves en la respuesta: 3/3 | se abstiene: False
FRAGMENTOS RECUPERADOS (* = citado en la respuesta):
 *1. [3] Stygian Blade - Aspects (similitud 0.796)
       Stygian Blade - Aspects: The Stygian Blade has 4 Aspects: Zagreus, Nemesis, Poseidon, and Arthur. Like all other weapons, each Aspect has 5 levels that can be unlocked by spending Titan Blood. The higher the level of the Aspect, the more effective its improvem
  2. [4] Stygian Blade - Aspects (similitud 0.797)
       Stygian Blade - Aspects: Aspect | Appearance | Description | Upgrades | Cost Aspect of Zagreus | The form in which the blade of the underworld first revealed itself. A blade such as this seeks not to remain idle for more than an aeon or two. | Bonus Attack Spe
  3. [6] Stygian Blade - Aspects (similitud 0.763)
       Stygian Blade - Aspec

### Puntuaciones y observaciones

Rellena una línea por pregunta, en el mismo orden de la tabla: (recuperacion, fidelidad, correccion, relevancia, trazabilidad, abstencion, observaciones). Usa números del 1 al 5 y "NA" cuando el criterio no se aplica. Las observaciones deben explicar la nota, sobre todo si es baja.

In [ ]:
PUNTUACIONES = [
    (5, 5, 5, 5, 5, "NA", "Los 4 aspectos están en el fragmento 3 (primero) y el resto del contexto es relevante."),
    (3, 5, 4, 5, 3, "NA", "Todo lo afirmado (Holy Excalibur, +50 de vida, aura de 8 s) está en 1860, pero la respuesta sale de una build del grupo: no se recupera la tabla wiki del aspecto (el fragmento 8 trata de desbloquearlo y el 532 es ruido). Los ids 3 y 1862 no respaldan lo afirmado."),
    (5, 5, 4, 5, 5, "NA", "Fragmento correcto en 1.ª posición. Dice que el ataque 'deals' crítico cuando la wiki dice 'may deal', y omite el bonus de 15% a 30%."),
    (5, 5, 5, 5, 5, "NA", "Fragmento 58 en 1.ª posición, respuesta exacta."),
    (4, 3, 3, 4, 4, "NA", "Fragmento 61 (descripción exacta) en 2.ª posición y 1245 en 4.ª. El dato de 10 a 5 s está en 1245. Pero el mecanismo queda confuso: dice 'cargar el siguiente ataque en el actual, disparando ambos', cuando la wiki dice que el Cast se carga en el siguiente Attack."),
    (4, 5, 5, 5, 5, "NA", "Hard Labor: +20% por rango, hasta +100%. El fragmento 1639 está en 2.ª posición con ruido del Pact."),
    (4, 5, 5, 5, 5, "NA", "Fragmento 871 en 1.ª posición, 25 encuentros correcto. El resto del contexto es ruido (llaveros)."),
    (5, 5, 5, 5, 5, "NA", "Fragmento 337 en 1.ª posición, 55 tareas correcto."),
    (5, 3, 4, 4, 4, "NA", "Definición, 28 Duo Boons y 8 dioses correctos (1121). La última frase distorsiona 1122: dice que 'adquirirlos' aumenta las probabilidades, cuando son los llaveros y talentos los que las suben. Información de más."),
    (5, 2, 1, 4, 3, "NA", "Fragmento 1286 en 1.ª posición, pero el modelo invierte el dato: dice que bajo el 66% hace 5, cuando sube a 16. Contradice su propio contexto."),
    (3, 5, 3, 5, 4, "NA", "398 (la lista de vías) está en 4.ª posición. Las cinco vías enumeradas están en 398, sin invenciones. Pero falta la fuente principal (Infernal Troves), que no se recupera, y se cita 1740 sin usarlo."),
    (5, 5, 4, 5, 5, "NA", "Fragmentos 1154, 1155 y 1156 en las primeras posiciones. Respuesta fiel a 1155. Omite que al completar el trial el dios perdona y ofrece su boon (1156)."),
    ("NA", 5, "NA", "NA", "NA", 4, "Se abstiene con la frase fija sin inventar. Añade ids [0] [1202] irrelevantes."),
    ("NA", 5, "NA", "NA", "NA", 4, "Se abstiene sin inventar. El fragmento 1201 no trae cifras de ventas. Añade una cita innecesaria."),
    ("NA", 5, "NA", "NA", "NA", 4, "Se abstiene sin inventar, pero cita cuatro ids sin relación (Loyalty Card, House Contractor...)."),
]

CRITERIOS = [c for c in evaluacion.columns if c.endswith("(1-5)")]

assert len(PUNTUACIONES) == len(evaluacion), "debe haber una línea por pregunta"
for i, (*notas, obs) in enumerate(PUNTUACIONES):
    evaluacion.loc[i, CRITERIOS] = notas
    evaluacion.loc[i, "observaciones"] = obs

### Revisión de la evaluación

Comprueba que aparecen las 15 preguntas, que los seis criterios tienen una puntuación válida (1 a 5 o NA) y que todas las preguntas tienen observaciones. Si todo está bien, muestra la media de cada criterio para las preguntas con respuesta y sin respuesta.

In [ ]:
def revisar_evaluacion(df, preguntas_esperadas=15):
    problemas = []
    if len(df) != preguntas_esperadas:
        problemas.append(f"hay {len(df)} preguntas y deberían ser {preguntas_esperadas}")
    if df["pregunta"].duplicated().any():
        problemas.append("hay preguntas repetidas")
    if len(CRITERIOS) != 6:
        problemas.append(f"hay {len(CRITERIOS)} criterios y deberían ser 6")
    for i, fila in df.iterrows():
        for c in CRITERIOS:
            v = fila[c]
            if v is None or (isinstance(v, float) and pd.isna(v)):
                problemas.append(f"pregunta {i + 1}: falta {c}")
            elif v != "NA" and not (isinstance(v, (int, float)) and 1 <= v <= 5):
                problemas.append(f"pregunta {i + 1}: valor no válido en {c}: {v!r}")
        if not str(fila["observaciones"]).strip():
            problemas.append(f"pregunta {i + 1}: sin observaciones")
    return problemas

problemas = revisar_evaluacion(evaluacion)
print("Preguntas en la tabla:", len(evaluacion))
print("Problemas encontrados:", len(problemas))
for p in problemas[:25]:
    print(" -", p)

if not problemas:
    numericas = evaluacion[CRITERIOS].replace("NA", np.nan).astype(float)
    display(numericas.groupby(evaluacion["esperada"]).mean().round(2))
    display(evaluacion[["pregunta", "esperada"] + CRITERIOS + ["observaciones"]])

Preguntas en la tabla: 15
Problemas encontrados: 0


/tmp/ipykernel_24634/316519719.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  numericas = evaluacion[CRITERIOS].replace("NA", np.nan).astype(float)


,recuperacion (1-5),fidelidad (1-5),correccion (1-5),relevancia (1-5),trazabilidad (1-5),abstencion (1-5)
esperada,,,,,,
con respuesta,4.42,4.42,4.0,4.75,4.42,NaN
sin respuesta,NaN,5.00,NaN,NaN,NaN,4.0


,pregunta,esperada,recuperacion (1-5),fidelidad (1-5),correccion (1-5),relevancia (1-5),trazabilidad (1-5),abstencion (1-5),observaciones
0,Which aspects does the Stygian Blade have?,con respuesta,5,5,5,5,5,NA,Los 4 aspectos están en el fragmento 3 (primero) y el resto del contexto es relevante.
1,What does the Aspect of Arthur give you?,con respuesta,3,5,4,5,3,NA,"Todo lo afirmado (Holy Excalibur, +50 de vida, aura de 8 s) está en 1860, pero la respuesta sale de una build del grupo: no se recupera la tabla wiki del aspecto (el fragmento 8 trata de desbloque..."
2,What does the Aspect of Nemesis do for the Stygian Blade?,con respuesta,5,5,4,5,5,NA,"Fragmento correcto en 1.ª posición. Dice que el ataque 'deals' crítico cuando la wiki dice 'may deal', y omite el bonus de 15% a 30%."
3,Which aspects does the Heart-Seeking Bow have?,con respuesta,5,5,5,5,5,NA,"Fragmento 58 en 1.ª posición, respuesta exacta."
4,What does the Aspect of Hera do for the Heart-Seeking Bow?,con respuesta,4,3,3,4,4,NA,"Fragmento 61 (descripción exacta) en 2.ª posición y 1245 en 4.ª. El dato de 10 a 5 s está en 1245. Pero el mecanismo queda confuso: dice 'cargar el siguiente ataque en el actual, disparando ambos'..."
5,What does the Hard Labor condition do in the Pact of Punishment?,con respuesta,4,5,5,5,5,NA,"Hard Labor: +20% por rango, hasta +100%. El fragmento 1639 está en 2.ª posición con ruido del Pact."
6,How many encounters does a keepsake need to reach rank 2?,con respuesta,4,5,5,5,5,NA,"Fragmento 871 en 1.ª posición, 25 encuentros correcto. El resto del contexto es ruido (llaveros)."
7,How many tasks are there in the Fated List of Minor Prophecies?,con respuesta,5,5,5,5,5,NA,"Fragmento 337 en 1.ª posición, 55 tareas correcto."
8,What are Duo Boons?,con respuesta,5,3,4,4,4,NA,"Definición, 28 Duo Boons y 8 dioses correctos (1121). La última frase distorsiona 1122: dice que 'adquirirlos' aumenta las probabilidades, cuando son los llaveros y talentos los que las suben. Inf..."
9,How much damage does the Bone Hydra's Lunge deal below 66% health?,con respuesta,5,2,1,4,3,NA,"Fragmento 1286 en 1.ª posición, pero el modelo invierte el dato: dice que bajo el 66% hace 5, cuando sube a 16. Contradice su propio contexto."


### Precision@K y Recall@K

Un fragmento se considera relevante si contiene las palabras clave de la pregunta. Precision@K es la fracción de los K fragmentos recuperados que son relevantes y Recall@K es la fracción de los fragmentos relevantes que se han recuperado.

Con solo uno o dos fragmentos relevantes por pregunta, Precision@5 no puede pasar de 0.2 o 0.4 aunque la recuperación sea perfecta. Por eso se interpreta junto con Recall@K, hit@k y MRR

In [ ]:

def metricas_recuperacion(k=5, modo="hibrido", tipos=("juego",)):
    filas = []

    for pregunta, claves in PRUEBAS_DENTRO:

        # Utilizamos los mismos documentos relevantes
        # que hemos definido en la sección anterior.
        relevantes = set(RELEVANTES_IDS[pregunta])

        # Recuperamos los K mejores documentos.
        resultados = recuperar(
            pregunta,
            k=k,
            modo=modo,
            tipos=tipos
        )

        recuperados = [
            r["documento"]["id"] for r in resultados
        ]

        # Documentos relevantes que realmente hemos encontrado.
        encontrados = relevantes.intersection(set(recuperados))

        aciertos = len(encontrados)

        precision = aciertos / k
        recall = aciertos / len(relevantes)

        filas.append({
            "pregunta": pregunta,
            "documentos relevantes": len(relevantes),
            "documentos encontrados": aciertos,
            f"precision@{k}": round(precision, 2),
            f"recall@{k}": round(recall, 2)
        })

    return pd.DataFrame(filas)


# Comparar las tres configuraciones
variantes = [
    ("Denso, todos los tipos", "denso", TODOS_LOS_TIPOS),
    ("Denso, solo juego", "denso", ("juego",)),
    ("Híbrido, solo juego", "hibrido", ("juego",))
]

resumenes = []

for nombre, modo, tipos in variantes:

    tabla_pk = metricas_recuperacion(
        k=5,
        modo=modo,
        tipos=tipos
    )

    resumenes.append({
        "variante": nombre,
        "Precision@5": round(
            float(tabla_pk["precision@5"].mean()), 2
        ),
        "Recall@5": round(
            float(tabla_pk["recall@5"].mean()), 2
        )
    })

    if nombre == "Híbrido, solo juego":
        detalle_hibrido = tabla_pk

resumen_pk = pd.DataFrame(resumenes)

print("COMPARACIÓN DE PRECISION@5 Y RECALL@5")
display(resumen_pk)

print("DETALLE DE LA BÚSQUEDA HÍBRIDA")
display(detalle_hibrido)


COMPARACIÓN DE PRECISION@5 Y RECALL@5


,variante,Precision@5,Recall@5
0,"Denso, todos los tipos",0.25,0.97
1,"Denso, solo juego",0.25,0.97
2,"Híbrido, solo juego",0.27,1.00


DETALLE DE LA BÚSQUEDA HÍBRIDA


,pregunta,documentos relevantes,documentos encontrados,precision@5,recall@5
0,Which aspects does the Stygian Blade have?,1,1,0.2,1.0
1,What does the Aspect of Arthur give you?,1,1,0.2,1.0
2,What does the Aspect of Nemesis do for the Stygian Blade?,3,3,0.6,1.0
3,Which aspects does the Heart-Seeking Bow have?,1,1,0.2,1.0
4,What does the Aspect of Hera do for the Heart-Seeking Bow?,2,2,0.4,1.0
5,What does the Hard Labor condition do in the Pact of Punishment?,1,1,0.2,1.0
6,How many encounters does a keepsake need to reach rank 2?,1,1,0.2,1.0
7,How many tasks are there in the Fated List of Minor Prophecies?,1,1,0.2,1.0
8,What are Duo Boons?,2,2,0.4,1.0
9,How much damage does the Bone Hydra's Lunge deal below 66% health?,1,1,0.2,1.0


### La abstención como clasificación

Se trata la abstención como un problema de clasificación: positivo = la pregunta no tiene respuesta y el sistema debe abstenerse. Hay 12 preguntas con respuesta y 3 sin respuesta, así que un sistema que nunca se abstuviera tendría una accuracy de 0.80. Por eso se miran también precision, recall y F1.

In [ ]:

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Normalizar texto para comparar las respuestas
def normalizar_respuesta(texto):
    return " ".join(
        str(texto).replace("’", "'").strip().lower().split()
    )


frase_no_info = normalizar_respuesta(NO_INFO)

# Etiquetas reales:
# 1 = pregunta sin respuesta documental
# 0 = pregunta con respuesta documental
y_real = (evaluacion["esperada"] == "sin respuesta").astype(int)

# Detectar si el asistente se abstiene.
# Puede añadir citas y seguir siendo una abstención.
y_pred = evaluacion["respuesta"].apply(
    lambda r: int(frase_no_info in normalizar_respuesta(r))
)

# Matriz de confusión
tn, fp, fn, tp = confusion_matrix(
    y_real, y_pred, labels=[0, 1]
).ravel()

# Métricas de clasificación
metricas_abstencion = {
    "Accuracy": round(accuracy_score(y_real, y_pred), 2),
    "Precision": round(precision_score(y_real, y_pred, zero_division=0), 2),
    "Recall": round(recall_score(y_real, y_pred, zero_division=0), 2),
    "F1": round(f1_score(y_real, y_pred, zero_division=0), 2)
}

print("=== EVALUACIÓN DE LA ABSTENCIÓN ===")
print(f"TP={tp} | TN={tn} | FP={fp} | FN={fn}")

for nombre, valor in metricas_abstencion.items():
    print(f"{nombre:12s}: {valor}")

# Comprobar si respeta exactamente la frase establecida
sin_respuesta = evaluacion[
    evaluacion["esperada"] == "sin respuesta"
].copy()

sin_respuesta["abstencion detectada"] = sin_respuesta["respuesta"].apply(
    lambda r: frase_no_info in normalizar_respuesta(r)
)

sin_respuesta["formato exacto"] = sin_respuesta["respuesta"].apply(
    lambda r: normalizar_respuesta(r) == frase_no_info
)

total_sin_respuesta = len(sin_respuesta)
formatos_correctos = int(sin_respuesta["formato exacto"].sum())

tasa_formato = (
    formatos_correctos / total_sin_respuesta
    if total_sin_respuesta > 0 else 0
)

print("\n=== CUMPLIMIENTO DEL FORMATO DE ABSTENCIÓN ===")
print(f"Abstenciones detectadas: {int(sin_respuesta['abstencion detectada'].sum())}/{total_sin_respuesta}")
print(f"Formato exacto: {formatos_correctos}/{total_sin_respuesta}")
print(f"Tasa de formato correcto: {tasa_formato:.2f}")

display(
    sin_respuesta[
        ["pregunta", "abstencion detectada", "formato exacto", "respuesta"]
    ]
)


=== EVALUACIÓN DE LA ABSTENCIÓN ===
TP=3 | TN=12 | FP=0 | FN=0
Accuracy    : 1.0
Precision   : 1.0
Recall      : 1.0
F1          : 1.0

=== CUMPLIMIENTO DEL FORMATO DE ABSTENCIÓN ===
Abstenciones detectadas: 3/3
Formato exacto: 0/3
Tasa de formato correcto: 0.00


,pregunta,abstencion detectada,formato exacto,respuesta
12,What is Melinoë's main weapon in Hades II?,True,False,I can't find that information in the available documents. [0] [1202]
13,How many copies of Hades has Supergiant Games sold worldwide?,True,False,I can't find that information in the available documents. [1201]
14,Which graphics card do I need to play Hades at 4K?,True,False,I can't find that information in the available documents. [895] [1201] [1202] [1359]


### Guardar la evaluación

Guarda la tabla en un archivo CSV y lo descarga, para conservar las puntuaciones aunque se cierre la sesión de Colab.

In [ ]:
evaluacion.to_csv("evaluacion.csv", index=False)
print("Guardado evaluacion.csv con", len(evaluacion), "preguntas")
try:
    from google.colab import files
    files.download("evaluacion.csv")
except Exception:
    pass

Guardado evaluacion.csv con 15 preguntas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## 13. Comparación de configuraciones de recuperación

Se comparan tres configuraciones del sistema utilizando las mismas 12 preguntas de evaluación.

Se analizan las métricas Hit@5, MRR, Precision@5 y Recall@5 para comprobar cómo afecta el tipo de búsqueda y el filtrado documental a la recuperación de información.

Esta comparación utiliza los resultados obtenidos en las evaluaciones anteriores, sin volver a generar respuestas con el modelo.


In [ ]:

# Unir los resultados de las métricas de recuperación
# obtenidos en las celdas anteriores.

recuperacion = tabla_recuperacion.rename(
    columns={"variante (k=5)": "variante"}
).copy()

precision_recall = resumen_pk.copy()

# Normalizar los nombres para poder unir las tablas.
recuperacion["clave"] = recuperacion["variante"].str.lower().str.strip()
precision_recall["clave"] = precision_recall["variante"].str.lower().str.strip()

comparacion = recuperacion.merge(
    precision_recall.drop(columns=["variante"]),
    on="clave",
    how="inner",
    validate="one_to_one"
)

# Mostrar las métricas de las tres configuraciones.
comparacion = comparacion[
    ["variante", "hit@k", "MRR", "Precision@5", "Recall@5"]
].rename(columns={
    "hit@k": "Hit@5"
})

print("=== COMPARACIÓN DE CONFIGURACIONES ===")
display(comparacion)


=== COMPARACIÓN DE CONFIGURACIONES ===


,variante,Hit@5,MRR,Precision@5,Recall@5
0,"denso, todos los tipos",1.0,0.85,0.25,0.97
1,"denso, solo juego",1.0,0.85,0.25,0.97
2,"híbrido, solo juego",1.0,0.75,0.27,1.00


## 14. Mejora de la memoria

Se repite un diálogo de dos turnos con y sin memoria. En el segundo turno la pregunta no nombra ni el arma ni el aspecto, así que sin memoria la búsqueda no puede encontrar el fragmento correcto.

In [ ]:
DIALOGO = [
    ("Which aspects does the Heart-Seeking Bow have?", ["Chiron", "Hera", "Rama"]),
    ("And what does the one named after Hera do?", ["Cast loads into your next Attack"]),
]

for memoria in (False, True):
    sistema = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=None, memoria=memoria)
    print("=" * 90)
    print("MEMORIA:", memoria)
    for pregunta, claves in DIALOGO:
        r = sistema.preguntar(pregunta)
        correcto = any(contiene_claves(x["documento"], claves) for x in r["recuperados"])
        print("\nQ:", pregunta)
        print("pregunta usada para buscar:", r["pregunta_autonoma"])
        print("A:", r["respuesta"])
        print("fragmento correcto recuperado:", correcto)

MEMORIA: False

Q: Which aspects does the Heart-Seeking Bow have?
pregunta usada para buscar: Which aspects does the Heart-Seeking Bow have?
A: The Heart-Seeking Bow has four aspects: Zagreus, Chiron, Hera, and Rama [58].
fragmento correcto recuperado: True

Q: And what does the one named after Hera do?
pregunta usada para buscar: And what does the one named after Hera do?
A: The Aspect of Hera allows for massive burst damage with a delayed reload time; when fully upgraded, it reduces the drop time of lodged enemies to 5 seconds instead of 10. [1245]
fragmento correcto recuperado: False
MEMORIA: True

Q: Which aspects does the Heart-Seeking Bow have?
pregunta usada para buscar: Which aspects does the Heart-Seeking Bow have?
A: The Heart-Seeking Bow has four aspects: Zagreus, Chiron, Hera, and Rama [58].
fragmento correcto recuperado: True

Q: And what does the one named after Hera do?
pregunta usada para buscar: What aspect does the Heart-Seeking Bow named after Hera represent?
A: The 

### Resumen de resultados para la memoria

Reúne en un solo bloque los números principales de la evaluación para copiarlos a la memoria. Interpretar los resultados (si cada mejora ha funcionado y por qué) es trabajo del grupo.

In [ ]:
print("=== RECUPERACIÓN (sección 10) ===")
display(tabla_recuperacion)
print("\n=== PRECISION@5 y RECALL@5 (denso, híbrido) ===")
print(resumen_pk)
print("\n=== UMBRAL DE SIMILITUD ===")
print("umbral fijado:", UMBRAL)
print("\n=== ABSTENCIÓN ===")
print(metricas_abstencion)
print("\n=== CUMPLIMIENTO DEL FORMATO DE ABSTENCIÓN ===")
print(f"Formato exacto: {formatos_correctos}/{total_sin_respuesta}")
print(f"Tasa de formato correcto: {tasa_formato:.2f}")
print("\n=== EVALUACIÓN MANUAL: media por criterio ===")
if not revisar_evaluacion(evaluacion):
    display(evaluacion[CRITERIOS].replace("NA", np.nan).astype(float).groupby(evaluacion["esperada"]).mean().round(2))
else:
    print("La evaluación manual aún no está completa (ejecuta la celda de revisión).")
print("\n=== COMPARACIÓN DE CONFIGURACIONES DE RECUPERACIÓN ===")
display(comparacion)

=== RECUPERACIÓN (sección 10) ===


,variante (k=5),hit@k,MRR
0,"denso, todos los tipos",1.0,0.85
1,"denso, solo juego",1.0,0.85
2,"híbrido, solo juego",1.0,0.75



=== PRECISION@5 y RECALL@5 (denso, híbrido) ===
                 variante  Precision@5  Recall@5
0  Denso, todos los tipos         0.25      0.97
1       Denso, solo juego         0.25      0.97
2     Híbrido, solo juego         0.27      1.00

=== UMBRAL DE SIMILITUD ===
umbral fijado: None

=== ABSTENCIÓN ===
{'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0}

=== CUMPLIMIENTO DEL FORMATO DE ABSTENCIÓN ===
Formato exacto: 0/3
Tasa de formato correcto: 0.00

=== EVALUACIÓN MANUAL: media por criterio ===


/tmp/ipykernel_24634/738378194.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(evaluacion[CRITERIOS].replace("NA", np.nan).astype(float).groupby(evaluacion["esperada"]).mean().round(2))


,recuperacion (1-5),fidelidad (1-5),correccion (1-5),relevancia (1-5),trazabilidad (1-5),abstencion (1-5)
esperada,,,,,,
con respuesta,4.42,4.42,4.0,4.75,4.42,NaN
sin respuesta,NaN,5.00,NaN,NaN,NaN,4.0



=== COMPARACIÓN DE CONFIGURACIONES DE RECUPERACIÓN ===


,variante,Hit@5,MRR,Precision@5,Recall@5
0,"denso, todos los tipos",1.0,0.85,0.25,0.97
1,"denso, solo juego",1.0,0.85,0.25,0.97
2,"híbrido, solo juego",1.0,0.75,0.27,1.00


## 15. Demostración interactiva

Conversación libre con memoria para la demostración en clase. Para terminar, escribe exit o deja la línea vacía. El comando reiniciar borra la memoria.

In [ ]:
demo = AsistenteHades(modo="hibrido", tipos=("juego",), umbral=UMBRAL, memoria=True)

while True:
    entrada = input("You: ").strip()
    if entrada.lower() in {"", "exit", "quit"}:
        break
    if entrada.lower() == "reiniciar":
        demo.reiniciar()
        print("(memoria borrada)")
        continue
    mostrar(demo.preguntar(entrada))
    print()

You: exit


## 16. Builds del grupo

La wiki no contiene builds, así que las escribe el grupo en builds.json (ver la plantilla builds_plantilla.json). Cada build se divide en 2 o 3 fragmentos: idea general, bendiciones y mejoras prioritarias, y puntos débiles.

Reglas para que se recuperen bien:

- Texto en inglés y con palabras propias.
- Cada fragmento nombra el arma, el aspecto y los dioses implicados, para entenderse por sí solo.
- Una sola idea por fragmento, de 40 a 120 palabras.
- Solo datos que hayáis comprobado en el juego o en la wiki.

Después de añadir o editar builds.json, vuelve a ejecutar desde la sección 3 (carga, embeddings, Qdrant y BM25).

## 17. Descripción, resultados y limitaciones

Temática y documentos: asistente sobre Hades (2018). La base documental procede de la wiki de Hades en Fandom (texto CC BY-SA, con la url de cada fragmento) y de las builds escritas por el grupo. Los fragmentos se organizan por entidad, sección, categoría y tipo.

Respuestas para la presentación:

- Organización de los documentos: fragmentos de una idea, con la entidad al inicio del texto, metadatos y filtro por tipo.
- Modelo de embeddings: BAAI/bge-base-en-v1.5, vectores normalizados y similitud coseno en Qdrant.
- Selección de documentos: búsqueda híbrida (densa y BM25) fusionada con RRF, filtro por tipo y K=5.
- Modelo generativo: Qwen/Qwen2.5-3B-Instruct, con plantilla de chat y generación determinista.
- Comprobación de fidelidad: prompt restrictivo, citas con el id del fragmento, fuentes mostradas y evaluación manual.
- Preguntas fuera de la colección: frase fija de abstención y umbral de similitud si la calibración lo permite.
- Siguiente versión: reranker, más preguntas de evaluación, más builds y revisión de los datos frente al juego.

Limitaciones: la wiki puede tener errores o datos cambiados con parches; el conjunto de pruebas es pequeño y la calibración del umbral es orientativa; un modelo de 3.000 millones de parámetros puede mezclar datos de fragmentos parecidos; las builds dependen de lo que escriba el grupo; las comprobaciones automáticas de palabras clave son aproximadas; las mismas 15 preguntas sirven para calibrar el umbral y para evaluar, así que el umbral y las métricas de abstención son orientativos; la puntuación manual es subjetiva, conviene indicar quién la ha hecho y cómo se resolvieron los desacuerdos.

Rellena con vuestros resultados reales las tablas de las secciones 10, 12, 13 y 14 y el resumen de resultados y comenta si cada mejora ha funcionado.

## 18. Conclusiones

Flujo implementado: documentos → embeddings → Qdrant → recuperación híbrida → contexto → LLM → respuesta con fuentes y memoria.

La calidad depende de la base documental, de cómo se fragmenta, del modelo de embeddings, de la recuperación y del modelo generativo. RAG reduce las invenciones pero no las elimina: hay que revisar siempre las respuestas y sus fuentes.